# NFXP applied workflow

A fleet manager must decide when to replace a component as condition worsens. This notebook fits a structural replacement model, checks identification and convergence, evaluates held-out choices, studies a lower replacement cost, and verifies that the fitted object can be reloaded.

In [ ]:
from importlib.metadata import version
import pickle

import numpy as np
from econirl import NFXP
from econirl.core.reward_spec import RewardSpec
from econirl.core.types import Panel
from econirl.environments import ArrayMDP
from econirl.simulation.synthetic import simulate_panel

print(f"EconIRL {version('econirl')}")

## Build and inspect the fleet panel

The state is a 12-level condition score. Action 0 keeps the component in service. Action 1 replaces it and resets condition. The reward uses two action-contrast features, so both coefficients can be identified from choices.

In [ ]:
n_states = 12
transitions = np.zeros((2, n_states, n_states), dtype=float)
for state in range(n_states):
    for increment, probability in ((0, 0.25), (1, 0.55), (2, 0.20)):
        transitions[0, state, min(state + increment, n_states - 1)] += probability
transitions[1, :, 0] = 1.0

condition = np.linspace(-1.0, 1.0, n_states)
features = np.zeros((n_states, 2, 2), dtype=float)
features[:, 1, 0] = condition
features[:, 1, 1] = 1.0
truth = np.array([1.1, -0.8])
env = ArrayMDP(
    transitions,
    features,
    theta=truth,
    discount_factor=0.90,
    parameter_names=["condition_pressure", "replacement_cost"],
    initial_distribution=np.ones(n_states) / n_states,
    seed=20260809,
)
full_panel = simulate_panel(env, n_individuals=80, n_periods=25, seed=20260809)
train = Panel(full_panel.trajectories[:60])
test = Panel(full_panel.trajectories[60:])
train_actions = np.asarray(train.get_all_actions())
print(f"Training observations: {train.num_observations}")
print(f"Training individuals: {train.num_individuals}")
print(f"Replacement share: {train_actions.mean():.3f}")
print(f"Held-out observations: {test.num_observations}")

## Fit, diagnose, and quantify uncertainty

The bootstrap resamples whole component histories. The summary reports the stopping reason and the assumptions that limit interpretation.

In [ ]:
reward = RewardSpec(features, names=["condition_pressure", "replacement_cost"])
model = NFXP(
    n_states=n_states,
    discount=0.90,
    utility=reward,
    se_method="bootstrap",
    n_bootstrap=29,
    se_seed=90210,
)
model.fit(train, transitions=transitions)
print(f"Action-contrast rank: {model.diagnostics_['identification']['contrast_rank']}")
print(f"Transition orientation: {model.diagnostics_['transitions']['orientation']}")
print()
print(model.summary())

## Check held-out choices

Held-out negative log likelihood evaluates new component histories that were not used for estimation.

In [ ]:
test_states = np.asarray(test.get_all_states(), dtype=int)
test_actions = np.asarray(test.get_all_actions(), dtype=int)
test_probabilities = model.predict_proba(test_states)
chosen = test_probabilities[np.arange(test_states.size), test_actions]
heldout_nll = -np.log(np.clip(chosen, 1e-15, 1.0)).mean()
print(f"Held-out negative log likelihood: {heldout_nll:.4f}")
print(f"Prediction rows sum to one: {np.allclose(test_probabilities.sum(axis=1), 1.0)}")

## Lower the replacement cost

The intervention raises the replacement utility coefficient by 0.5. NFXP re-solves the dynamic program under the fitted reward change.

In [ ]:
changed_cost = model.params_["replacement_cost"] + 0.5
counterfactual = model.counterfactual(replacement_cost=changed_cost)
baseline_rate = model.policy_[:, 1].mean()
changed_rate = counterfactual.counterfactual_policy[:, 1].mean()
mean_policy_change = np.abs(
    counterfactual.counterfactual_policy - counterfactual.baseline_policy
).mean()
print(f"Mean replacement probability before: {baseline_rate:.3f}")
print(f"Mean replacement probability after: {changed_rate:.3f}")
print(f"Mean absolute policy change: {mean_policy_change:.3f}")

## Reload the fitted estimator

A pickle round trip preserves the summary, predictions, and counterfactual inputs within the same EconIRL minor release.

In [ ]:
restored = pickle.loads(pickle.dumps(model))
prediction_gap = np.max(
    np.abs(restored.predict_proba(np.arange(n_states)) - model.predict_proba(np.arange(n_states)))
)
print(f"Stored EconIRL version: {restored.econirl_version_}")
print(f"Summary preserved: {restored.summary() == model.summary()}")
print(f"Maximum prediction gap: {prediction_gap:.1e}")